# Física BR · Taller: Corriente, Resistencia y Resistividad
### Física BR — Física de Electricidad y Magnetismo
**Profesor:** Juan David Betancur Ríos · **Semestre:** 2026-2

---
*Física interactiva para ingeniería.* La corriente eléctrica, la resistencia y la resistividad
son la base del análisis de circuitos y del diseño eléctrico. Cada punto parte de un problema real
de ingeniería e incluye esquema, sliders, botón **"Mostrar solución"** y cuestionario.

**Convenciones:** tres cifras significativas · coma decimal · carga del electrón e = 1,60×10⁻¹⁹ C.

**Ecuaciones clave:**
- Corriente:  I = Q/t
- Ley de Ohm:  V = I·R
- Resistividad:  R = ρ·L/A    (ρ del material, L longitud, A área de sección)
- Potencia eléctrica:  P = V·I = I²·R = V²/R
- Variación con temperatura:  R(T) = R₀·(1 + α·ΔT)

> **Uso:** `Entorno de ejecución → Ejecutar todo`. Competencias: [CL, UFT, UCM, PRC, ULC].

---
> © 2026 **Juan David Betancur Ríos** (docente de física para ingeniería) y **Nora Yamile Rojas Cataño** (aporte pedagógico y metodología de evaluación). Material educativo de uso académico (Física BR). Todos los derechos reservados.
> *Elaborado con apoyo de herramientas de inteligencia artificial, bajo la supervisión y criterio pedagógico de los autores.*

---
## Punto 1 — Corriente en un conductor (Ing. Eléctrica/Electrónica)  ★★  [CL, UCM]
Por la sección de un conductor pasa una carga  Q = 60,0 C  en un tiempo  t = 30,0 s.

**Corriente:**  I = Q/t    **Número de electrones:**  N = Q/e   (e = 1,60×10⁻¹⁹ C)

> 🔧 **Aplicación en ingeniería:** la corriente mide el flujo de carga y determina el calibre de
> los conductores (RETIE) y la capacidad de los dispositivos. Subdimensionar un cable para la
> corriente que circula provoca sobrecalentamiento: por eso el cálculo de I es el primer paso.

> 🤖 **TIP-IA — Verificador crítico:**
> Antes de mirar la solución, dile a un asistente: *"Calculé la corriente I=Q/t y el número de electrones N=Q/e. Actúa como evaluador escéptico: pregúntame si al aumentar el tiempo (misma carga) la corriente sube o baja, y por qué. No me des la respuesta."* Primero resuelves tú, después consultas.


In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display

def _3c(x):
    # 3 cifras significativas con coma decimal; ×10ⁿ fuera de 0,001 ≤ |x| < 1000 (textos de gráficas)
    import math
    try: x = float(x)
    except (TypeError, ValueError): return str(x)
    if x == 0 or not math.isfinite(x): return str(x).replace('.', ',') if x else '0'
    x = float(f"{x:.2e}")
    if abs(x) >= 1000 or abs(x) < 0.001:
        e = int(math.floor(math.log10(abs(x)))); m = x / 10**e
        if round(abs(m), 2) >= 10: m /= 10; e += 1
        return f"{m:.2f}".replace('.', ',').replace('-', '−') + '×10' + str(e).translate(str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹'))
    t = f"{x:#.3g}"
    return (t[:-1] if t.endswith('.') else t).replace('.', ',').replace('-', '−')
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

try:
    print_es
except NameError:
    def print_es(*args, **kwargs):
        import re as _re
        print(*[_re.sub(r"(\d)\.(\d)", r"\1,\2", a) if isinstance(a, str) else a for a in args], **kwargs)
# Signo menos tipográfico (−) en todo texto impreso: «-5» → «−5» (no toca guiones entre palabras, rangos ni 1e-3)
import builtins as _bi_br
if not getattr(_bi_br.print, '_menos_br', False):
    def _print_br(*a, _orig=_bi_br.print, **k):
        import re as _re
        _orig(*[_re.sub(r'(?<![\w.,\-])-(?=\d)', '−', x) if isinstance(x, str) else x for x in a], **k)
    _print_br._menos_br = True
    _bi_br.print = _print_br
# --- Cuestionarios Física BR v3: opciones barajadas, crédito por intento, nota sobre el total ---
import random as _rnd_br
try:
    PESOS_BR
except NameError:
    PESOS_BR = {}      # id_preg -> peso de cada pregunta mostrada
try:
    INTENTOS_BR
except NameError:
    INTENTOS_BR = {}   # id_preg -> [intentos, resuelta]
_CREDITO_BR = {1: 1.0, 2: 0.5, 3: 0.25}   # crédito según el intento en que se acierta
_MAX_INT_BR = 3

def _evaluar_intento(id_preg, ok, peso, explicacion, msg_ok, msg_mal):
    est = INTENTOS_BR.setdefault(id_preg, [0, False])
    if est[1]:
        print("✔ Ya resolviste esta pregunta; tu crédito quedó registrado.")
        if explicacion: print_es("→ " + explicacion)
        return
    if est[0] >= _MAX_INT_BR:
        print("⛔ Agotaste los 3 intentos con crédito: esta pregunta ya no suma.")
        if explicacion: print_es("→ " + explicacion)
        return
    est[0] += 1
    n = est[0]
    if ok:
        est[1] = True
        cred = _CREDITO_BR.get(n, 0.0)
        REGISTRO_BR[id_preg] = (cred, float(peso))
        print_es(msg_ok + f"  (intento {n}: crédito {int(cred*100)} %)")
        if explicacion: print_es("→ " + explicacion)
    else:
        REGISTRO_BR.setdefault(id_preg, (0.0, float(peso)))
        if n < _MAX_INT_BR:
            print(msg_mal + f"  Intento {n} de {_MAX_INT_BR}. Revisa tu razonamiento antes de volver a intentar.")
        else:
            print(msg_mal + " Agotaste los 3 intentos: esta pregunta ya no suma. Estudia la explicación:")
            if explicacion: print_es("→ " + explicacion)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    orden = list(range(len(opciones))); _rnd_br.shuffle(orden)   # orden distinto en cada sesión
    correcta = orden.index(indice_correcto)
    letras = "abcdefgh"
    etiquetas = [f"{letras[i]}) {opciones[j]}" for i, j in enumerate(orden)]
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=etiquetas, value=None, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            if radio.index is None:
                print("⚠ Elige una opción."); return
            _evaluar_intento(id_preg, radio.index == correcta, peso, explicacion,
                             "✅ ¡Correcto!", "❌ Incorrecto.")
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.strip().replace(",", "."))
            except ValueError: print("⚠ Escribe un número (usa coma o punto decimal)."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            _evaluar_intento(id_preg, ok, peso, explicacion,
                             f"✅ ¡Correcto!  ({valor_correcto:.3g} {unidad})", "❌ Fuera de rango.")
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not PESOS_BR:
                print("No se encontraron cuestionarios en este taller."); return
            peso_total = sum(PESOS_BR.values())
            peso_ok = sum(float(c)*p for c, p in REGISTRO_BR.values())
            nota = 5.0*peso_ok/peso_total if peso_total > 0 else 0.0
            total = len(PESOS_BR)
            resp = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0])[0] > 0)
            prim = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0, False])[1] and INTENTOS_BR[k][0] == 1)
            print("="*46)
            print("          CALIFICACIÓN FORMATIVA")
            print("="*46)
            print(f"  Preguntas respondidas:       {resp} de {total}")
            print(f"  Acertadas al primer intento: {prim}")
            print_es(f"  Puntaje ponderado:           {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*46)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*46)
            if resp < total: print(f"  ⚠ Faltan {total-resp} preguntas: cuentan como 0.")
            if nota >= 3.5: print("  ✅ ¡Buen trabajo! Dominas los conceptos.")
            elif nota >= 3.0: print("  🟡 Vas bien, repasa lo que fallaste.")
            else: print("  🔴 Repasa la teoría y vuelve a los puntos.")
            print("\n  Crédito por pregunta: 100 % al primer intento,")
            print("  50 % al segundo, 25 % al tercero. Predice y")
            print("  razona ANTES de verificar: adivinar cuesta nota.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>📊 Tu calificación del taller</h3>"
        "<i>La nota se calcula sobre TODAS las preguntas del taller y premia acertar al primer intento.</i>"),
        boton, salida]))


eps=1.60e-19
def esquema():
    fig,ax=plt.subplots(figsize=(7.5,3.5))
    # conductor
    ax.add_patch(plt.Rectangle((0,1),8,1.5,fill=False,ec='#2c3e50',lw=2))
    ax.fill_between([0,8],1,2.5,color='#f39c12',alpha=0.08)
    # sección transversal (línea que cuentan los electrones)
    ax.plot([4,4],[1,2.5],color='#c0392b',lw=2,ls='--')
    ax.text(4,2.8,'sección',ha='center',fontsize=8,color='#c0392b')
    # electrones fluyendo
    for x in [0.8,1.8,2.8,5.2,6.2,7.2]:
        ax.scatter([x],[1.75],s=100,color='#2980b9',zorder=5,edgecolor='white',linewidth=1)
        ax.annotate('',xy=(x+0.7,1.75),xytext=(x+0.3,1.75),arrowprops=dict(arrowstyle='-|>',color='#2980b9',lw=1.5))
    ax.text(4,0.4,'Corriente I = carga que cruza la sección por segundo = Q/t',
            ha='center',fontsize=9,color='#2c3e50',fontweight='bold')
    ax.text(1,3.1,'flujo de electrones →',fontsize=9,color='#2980b9',fontweight='bold')
    ax.set_xlim(-0.5,8.5); ax.set_ylim(0,3.4); ax.axis('off')
    ax.set_title('Corriente eléctrica: flujo de carga')
    plt.tight_layout(); plt.show()
esquema()

def corriente(Q=60.0, t=30.0):
    I=Q/t
    ts=np.linspace(5,60,100); Is=Q/ts
    plt.figure(figsize=(9,4.2)); plt.plot(ts,Is,lw=2,color='darkred')
    plt.scatter([t],[I],color='blue',s=60,zorder=5,label=f't={t:.0f} s → I={_3c(I)} A')
    plt.xlabel('tiempo t (s)'); plt.ylabel('corriente I (A)')
    plt.title('Corriente vs tiempo (I=Q/t)'); plt.legend(); plt.grid(alpha=.3)
    plt.tight_layout(); plt.show()


def solucion():
    import re as _re
    def _pc(*a,**k):
        def _f(t):
            if not isinstance(t,str): return t
            _sup='⁰¹²³⁴⁵⁶⁷⁸⁹'
            def _sci(m):
                mant=m.group(1); exp=int(m.group(2))
                e=('⁻' if exp<0 else '')+''.join(_sup[int(d)] for d in str(abs(exp)))
                return mant+'×10'+e
            t=_re.sub(r'(\d+(?:\.\d+)?)[eE]([+-]?\d+)', _sci, t)
            _u=r'(?:s|km/s|km|m/s²|m/s|m|N·m|N|J|W|A|V|T|Ω|C|F|Hz|rad/s²|rad/s|rad|kg·m²/s|kg·m²|kg|Pa|kPa|µF|nC|µC|pF|mWb|Wb|µT|mT|kW|MW|mA|kΩ|MΩ|rpm|°)'
            def _c3(m):
                num=m.group(2); n=int(num)
                if n==0: return m.group(0)
                if len(num)==1: nv=num+'.00'
                elif len(num)==2: nv=num+'.0'
                else: nv=num
                return m.group(1)+nv+m.group(3)
            t=_re.sub(r'(?<![\d.,])([\s=(×])(\d{1,3})(\s'+_u+r'\b)', _c3, t)
            t=_re.sub(r'(\d)\.(\d)', r'\1,\2', t)
            return t
        print(*[_f(x) for x in a], **k)

    Q=60.0; t=30.0; e=1.60e-19
    _pc(f"Corriente: I = Q/t = 60,0/30,0 = {Q/t:.3g} A")
    _pc(f"Número de electrones: N = Q/e = 60,0/1,60e-19 = {Q/e:.3g}")
    _pc("La corriente es el flujo de carga por unidad de tiempo (1 A = 1 C/s).")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 1 ---")
quiz_numerico("Por la sección de un conductor pasa una carga de 60,0 C en un tiempo de 30,0 s. Encuentre la corriente eléctrica.",2.0,0.04,"A","I=Q/t=2,00 A.", id_preg="Corrie_p1_1", peso=2.0)
quiz_opcion_multiple("En el Sistema Internacional, la unidad de corriente (el amperio) equivale a:",
    ['1 V/Ω (por la ley de Ohm)', '1 C/s (coulomb por segundo)', '1 J/s (energía por tiempo)', '1 Ω·s (resistencia por tiempo)'],1,"I=Q/t ⟹ 1 A = 1 C/s.", id_preg="Corrie_p1_2", peso=2.0)

# --- Estilo visual Física BR (apariencia global de los esquemas) ---
try:
    import matplotlib.pyplot as _plt_style
    _plt_style.rcParams['axes.spines.top']=False
    _plt_style.rcParams['axes.spines.right']=False
    _plt_style.rcParams['axes.titlesize']=12
    _plt_style.rcParams['axes.titleweight']='bold'
    _plt_style.rcParams['axes.titlecolor']='#2c3e50'
    _plt_style.rcParams['axes.labelsize']=10
    _plt_style.rcParams['axes.labelcolor']='#34495e'
    _plt_style.rcParams['font.size']=10
    _plt_style.rcParams['axes.grid']=True
    _plt_style.rcParams['grid.alpha']=0.25
    _plt_style.rcParams['grid.linestyle']='--'
    _plt_style.rcParams['axes.edgecolor']='#7f8c8d'
    _plt_style.rcParams['xtick.color']='#34495e'
    _plt_style.rcParams['ytick.color']='#34495e'
    _plt_style.rcParams['figure.facecolor']='white'
    _plt_style.rcParams['axes.facecolor']='#fcfcfd'
except Exception:
    pass
# --- fin estilo ---

quiz_opcion_multiple("DEFIENDE TU CONCLUSIÓN: un estudiante dice que si pasa la MISMA carga en MÁS tiempo, la corriente es mayor porque 'el conductor trabaja durante más tiempo'. ¿Correcto o solo convincente?",
    ['Solo convincente: I = Q/t; con la misma carga repartida en más tiempo, la corriente es menor',
     'Correcto: la corriente mide cuánto trabaja el conductor, y trabajar más tiempo es más corriente',
     'Correcto: la corriente es carga por tiempo, I = Q·t, así que más tiempo da más corriente',
     'Solo convincente: la corriente no cambia, porque la carga total que pasa es la misma'],
    0,
    'La corriente es un RITMO (carga por segundo), no un total. La misma carga en el doble de tiempo es la mitad de corriente. La carga total igual no implica corriente igual.',
    id_preg='Corr_p1_concep_1', peso=2.0)

def _interact_coma(fn, _crudos=(), **controles):
    # Como interact, con lectura de sliders en coma decimal. La salida NO se captura con «with salida:»:
    # se calculan el texto y las figuras y se escriben con append_stdout/append_display_data
    # (evita bloqueos de Voilà en Binder y funciona también en JupyterLite).
    import ipywidgets as _w, io as _io, contextlib as _cl
    import matplotlib.pyplot as _plt
    def _fmt(v, paso, crudo=False):
        # Lectura con coma decimal y 3 cifras significativas (×10ⁿ fuera de 0,001 ≤ |v| < 1000).
        # IntSlider: entero exacto. «crudo»: decimales del paso (slider que representa una medición sin redondear).
        import math
        if isinstance(v, int) and not isinstance(v, bool): return str(v)
        if crudo:
            s = repr(float(paso)); dec = 0 if 'e' in s else len(s.split('.')[1].rstrip('0'))
            return f"{v:.{min(dec, 4)}f}".replace('.', ',').replace('-', '−')
        v = float(v)
        if v == 0 or not math.isfinite(v): return '0' if v == 0 else str(v)
        v = float(f"{v:.2e}")
        if abs(v) >= 1000 or abs(v) < 0.001:
            m, e = f"{v:.2e}".split('e')
            return m.replace('.', ',').replace('-', '−') + '×10' + str(int(e)).translate(str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹'))
        t = f"{v:#.3g}"
        return (t[:-1] if t.endswith('.') else t).replace('.', ',').replace('-', '−')
    filas = []
    for nombre, c in controles.items():
        if isinstance(c, (_w.FloatSlider, _w.IntSlider)):
            c.readout = False
            lab = _w.Label(layout=_w.Layout(min_width='90px'))
            def _act(*_, c=c, lab=lab, cr=(nombre in _crudos)): lab.value = _fmt(c.value, c.step, cr)
            c.observe(_act, 'value'); _act()
            filas.append(_w.HBox([c, lab]))
        else:
            filas.append(c)
    salida = _w.Output()
    def _ejecutar(*_):
        partes, buf = [], _io.StringIO()
        def _vaciar_texto():
            t = buf.getvalue()
            if t: partes.append(('t', t)); buf.seek(0); buf.truncate(0)
        def _capturar(*a, **k):
            _vaciar_texto()
            for num in _plt.get_fignums():
                b = _io.BytesIO(); _plt.figure(num).savefig(b, format='png', bbox_inches='tight')
                partes.append(('i', b.getvalue()))
            _plt.close('all')
        _show = _plt.show; _plt.show = _capturar
        try:
            with _cl.redirect_stdout(buf):
                try: fn(**{k: c.value for k, c in controles.items()})
                except Exception as e: print('⚠ No se pudo calcular con estos valores:', e)
            _capturar()
        finally:
            _plt.show = _show
        _vaciar_texto()
        # Reemplazo en UNA sola asignación: la salida nunca queda vacía (sin saltos de la página)
        import base64 as _b64
        nuevas = []
        for tipo, val in partes:
            if tipo == 't': nuevas.append({'output_type': 'stream', 'name': 'stdout', 'text': val})
            else: nuevas.append({'output_type': 'display_data', 'metadata': {},
                                 'data': {'image/png': _b64.b64encode(val).decode('ascii'), 'text/plain': '<Figure>'}})
        salida.outputs = tuple(nuevas)
    for c in controles.values():
        c.observe(_ejecutar, 'value')
    _ejecutar()
    display(_w.VBox(filas + [salida]))

print("\n--- 🔬 Explora después de responder ---")
print("Los controles NO arrancan en los datos del enunciado. Primero resuelve y responde;\nluego explora tendencias, casos límite y comprueba tu razonamiento.")
_interact_coma(corriente,
    Q=FloatSlider(value=90.0,min=10,max=120,step=10,description='Q (C)'),
    t=FloatSlider(value=45.0,min=5,max=60,step=5,description='t (s)'));

---
## Punto 2 — Ley de Ohm en un circuito (Ing. Eléctrica/Electrónica)  ★★  [CL, UFT]
Una resistencia de  R = 60,0 Ω  se conecta a una fuente de  V = 120 V  (tensión típica de red).

**Ley de Ohm:**  I = V/R    **Potencia:**  P = V·I

> 🔧 **Aplicación en ingeniería:** la ley de Ohm es la relación fundamental para analizar cualquier
> circuito. Conocer la corriente que consume un dispositivo permite dimensionar protecciones
> (breakers, fusibles) y verificar que los conductores soporten esa corriente sin sobrecalentarse.

> 🤖 **TIP-IA — Abogado del diablo:**
> Dile a un asistente: *"Voy a sostener que si aumento la resistencia de un circuito, la corriente también aumenta porque 'hay más material para conducir'. Dame el mejor contraargumento."* (Pista: I=V/R, la R está en el denominador). La IA es espejo de tu razonamiento; primero decide tú.


In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display

def _3c(x):
    # 3 cifras significativas con coma decimal; ×10ⁿ fuera de 0,001 ≤ |x| < 1000 (textos de gráficas)
    import math
    try: x = float(x)
    except (TypeError, ValueError): return str(x)
    if x == 0 or not math.isfinite(x): return str(x).replace('.', ',') if x else '0'
    x = float(f"{x:.2e}")
    if abs(x) >= 1000 or abs(x) < 0.001:
        e = int(math.floor(math.log10(abs(x)))); m = x / 10**e
        if round(abs(m), 2) >= 10: m /= 10; e += 1
        return f"{m:.2f}".replace('.', ',').replace('-', '−') + '×10' + str(e).translate(str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹'))
    t = f"{x:#.3g}"
    return (t[:-1] if t.endswith('.') else t).replace('.', ',').replace('-', '−')
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

try:
    print_es
except NameError:
    def print_es(*args, **kwargs):
        import re as _re
        print(*[_re.sub(r"(\d)\.(\d)", r"\1,\2", a) if isinstance(a, str) else a for a in args], **kwargs)
# Signo menos tipográfico (−) en todo texto impreso: «-5» → «−5» (no toca guiones entre palabras, rangos ni 1e-3)
import builtins as _bi_br
if not getattr(_bi_br.print, '_menos_br', False):
    def _print_br(*a, _orig=_bi_br.print, **k):
        import re as _re
        _orig(*[_re.sub(r'(?<![\w.,\-])-(?=\d)', '−', x) if isinstance(x, str) else x for x in a], **k)
    _print_br._menos_br = True
    _bi_br.print = _print_br
# --- Cuestionarios Física BR v3: opciones barajadas, crédito por intento, nota sobre el total ---
import random as _rnd_br
try:
    PESOS_BR
except NameError:
    PESOS_BR = {}      # id_preg -> peso de cada pregunta mostrada
try:
    INTENTOS_BR
except NameError:
    INTENTOS_BR = {}   # id_preg -> [intentos, resuelta]
_CREDITO_BR = {1: 1.0, 2: 0.5, 3: 0.25}   # crédito según el intento en que se acierta
_MAX_INT_BR = 3

def _evaluar_intento(id_preg, ok, peso, explicacion, msg_ok, msg_mal):
    est = INTENTOS_BR.setdefault(id_preg, [0, False])
    if est[1]:
        print("✔ Ya resolviste esta pregunta; tu crédito quedó registrado.")
        if explicacion: print_es("→ " + explicacion)
        return
    if est[0] >= _MAX_INT_BR:
        print("⛔ Agotaste los 3 intentos con crédito: esta pregunta ya no suma.")
        if explicacion: print_es("→ " + explicacion)
        return
    est[0] += 1
    n = est[0]
    if ok:
        est[1] = True
        cred = _CREDITO_BR.get(n, 0.0)
        REGISTRO_BR[id_preg] = (cred, float(peso))
        print_es(msg_ok + f"  (intento {n}: crédito {int(cred*100)} %)")
        if explicacion: print_es("→ " + explicacion)
    else:
        REGISTRO_BR.setdefault(id_preg, (0.0, float(peso)))
        if n < _MAX_INT_BR:
            print(msg_mal + f"  Intento {n} de {_MAX_INT_BR}. Revisa tu razonamiento antes de volver a intentar.")
        else:
            print(msg_mal + " Agotaste los 3 intentos: esta pregunta ya no suma. Estudia la explicación:")
            if explicacion: print_es("→ " + explicacion)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    orden = list(range(len(opciones))); _rnd_br.shuffle(orden)   # orden distinto en cada sesión
    correcta = orden.index(indice_correcto)
    letras = "abcdefgh"
    etiquetas = [f"{letras[i]}) {opciones[j]}" for i, j in enumerate(orden)]
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=etiquetas, value=None, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            if radio.index is None:
                print("⚠ Elige una opción."); return
            _evaluar_intento(id_preg, radio.index == correcta, peso, explicacion,
                             "✅ ¡Correcto!", "❌ Incorrecto.")
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.strip().replace(",", "."))
            except ValueError: print("⚠ Escribe un número (usa coma o punto decimal)."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            _evaluar_intento(id_preg, ok, peso, explicacion,
                             f"✅ ¡Correcto!  ({valor_correcto:.3g} {unidad})", "❌ Fuera de rango.")
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not PESOS_BR:
                print("No se encontraron cuestionarios en este taller."); return
            peso_total = sum(PESOS_BR.values())
            peso_ok = sum(float(c)*p for c, p in REGISTRO_BR.values())
            nota = 5.0*peso_ok/peso_total if peso_total > 0 else 0.0
            total = len(PESOS_BR)
            resp = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0])[0] > 0)
            prim = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0, False])[1] and INTENTOS_BR[k][0] == 1)
            print("="*46)
            print("          CALIFICACIÓN FORMATIVA")
            print("="*46)
            print(f"  Preguntas respondidas:       {resp} de {total}")
            print(f"  Acertadas al primer intento: {prim}")
            print_es(f"  Puntaje ponderado:           {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*46)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*46)
            if resp < total: print(f"  ⚠ Faltan {total-resp} preguntas: cuentan como 0.")
            if nota >= 3.5: print("  ✅ ¡Buen trabajo! Dominas los conceptos.")
            elif nota >= 3.0: print("  🟡 Vas bien, repasa lo que fallaste.")
            else: print("  🔴 Repasa la teoría y vuelve a los puntos.")
            print("\n  Crédito por pregunta: 100 % al primer intento,")
            print("  50 % al segundo, 25 % al tercero. Predice y")
            print("  razona ANTES de verificar: adivinar cuesta nota.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>📊 Tu calificación del taller</h3>"
        "<i>La nota se calcula sobre TODAS las preguntas del taller y premia acertar al primer intento.</i>"),
        boton, salida]))


def esquema():
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(9,4),gridspec_kw={'width_ratios':[1,1.1]})
    # IZQUIERDA: circuito simple
    ax1.add_patch(plt.Rectangle((1,1),4,2.5,fill=False,ec='#2c3e50',lw=2))
    # batería
    ax1.plot([1,1],[1.8,2.7],color='#2c3e50',lw=1)
    ax1.plot([0.8,1.2],[2.7,2.7],color='#c0392b',lw=4); ax1.plot([0.85,1.15],[1.8,1.8],color='#2980b9',lw=2)
    ax1.text(0.4,2.25,'V',fontsize=11,color='#2c3e50',fontweight='bold')
    # resistencia (zigzag)
    xr=np.linspace(2,4,20); yr=3.5+0.12*np.sin(np.linspace(0,6*np.pi,20))
    ax1.plot(xr,yr,color='#e67e22',lw=2); ax1.text(3,3.9,'R',fontsize=11,color='#e67e22',fontweight='bold',ha='center')
    # corriente
    ax1.annotate('',xy=(3.5,1),xytext=(2.5,1),arrowprops=dict(arrowstyle='-|>',color='#27ae60',lw=2))
    ax1.text(3,0.6,'I',fontsize=11,color='#27ae60',fontweight='bold',ha='center')
    ax1.set_xlim(0,5.5); ax1.set_ylim(0,4.3); ax1.axis('off')
    ax1.set_title('Circuito básico',fontsize=10)
    # DERECHA: V vs I lineal (pendiente = R)
    I=np.linspace(0,4,100); V=2*I
    ax2.plot(I,V,color='#8e44ad',lw=2.5)
    ax2.text(2,2,'pendiente = R',fontsize=11,color='#8e44ad',fontweight='bold',rotation=32)
    ax2.set_xlabel('corriente I (A)'); ax2.set_ylabel('voltaje V (V)')
    ax2.set_title('V = I·R (relación lineal)',fontsize=10)
    ax2.grid(alpha=0.2,linestyle='--')
    plt.suptitle('Ley de Ohm: el voltaje impulsa la corriente contra la resistencia',fontsize=11,fontweight='bold',color='#2c3e50')
    plt.tight_layout(); plt.show()
esquema()

def ohm(V=120.0, R=60.0):
    I=V/R; P=V*I
    Rs=np.linspace(10,200,100); Is=V/Rs
    plt.figure(figsize=(9,4.2)); plt.plot(Rs,Is,lw=2,color='navy')
    plt.scatter([R],[I],color='red',s=60,zorder=5,label=f'R={R:.0f} Ω → I={_3c(I)} A, P={P:.0f} W')
    plt.xlabel('resistencia R (Ω)'); plt.ylabel('corriente I (A)')
    plt.title('Ley de Ohm: I=V/R (a mayor R, menor I)'); plt.legend(); plt.grid(alpha=.3)
    plt.tight_layout(); plt.show()


def solucion():
    import re as _re
    def _pc(*a,**k):
        def _f(t):
            if not isinstance(t,str): return t
            _sup='⁰¹²³⁴⁵⁶⁷⁸⁹'
            def _sci(m):
                mant=m.group(1); exp=int(m.group(2))
                e=('⁻' if exp<0 else '')+''.join(_sup[int(d)] for d in str(abs(exp)))
                return mant+'×10'+e
            t=_re.sub(r'(\d+(?:\.\d+)?)[eE]([+-]?\d+)', _sci, t)
            _u=r'(?:s|km/s|km|m/s²|m/s|m|N·m|N|J|W|A|V|T|Ω|C|F|Hz|rad/s²|rad/s|rad|kg·m²/s|kg·m²|kg|Pa|kPa|µF|nC|µC|pF|mWb|Wb|µT|mT|kW|MW|mA|kΩ|MΩ|rpm|°)'
            def _c3(m):
                num=m.group(2); n=int(num)
                if n==0: return m.group(0)
                if len(num)==1: nv=num+'.00'
                elif len(num)==2: nv=num+'.0'
                else: nv=num
                return m.group(1)+nv+m.group(3)
            t=_re.sub(r'(?<![\d.,])([\s=(×])(\d{1,3})(\s'+_u+r'\b)', _c3, t)
            t=_re.sub(r'(\d)\.(\d)', r'\1,\2', t)
            return t
        print(*[_f(x) for x in a], **k)

    V=120.0; R=60.0; I=V/R
    _pc(f"Corriente: I = V/R = 120/60,0 = {I:.3g} A")
    _pc(f"Potencia disipada: P = V·I = 120·{I:.1f} = {V*I:.3g} W")
    _pc("A mayor resistencia, menor corriente para el mismo voltaje.")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 2 ---")
quiz_numerico("Una resistencia de 60,0 Ω se conecta a una fuente de 120 V. Encuentre la corriente que circula por ella.",2.0,0.04,"A","I=V/R=2,00 A.", id_preg="Corrie_p2_3", peso=2.0)
quiz_opcion_multiple("Por la ley de Ohm (V=IR), si se mantiene el mismo voltaje pero se DUPLICA la resistencia, ¿qué ocurre con la corriente?",['Se duplica (I ∝ R)', 'Se reduce a la mitad (I ∝ 1/R)', 'No cambia (V es fijo)', 'Se cuadruplica (I ∝ R²)'],1,"Con V fijo, I=V/R: la corriente es inversamente proporcional a la resistencia. Al duplicar R, la corriente cae a la mitad. Más resistencia opone más al paso de la corriente.", id_preg="Corrie_p2_4", peso=2.0)
quiz_opcion_multiple("Según la ley de Ohm, si se duplica la resistencia manteniendo el mismo voltaje, ¿qué ocurre con la corriente?",
    ['Se duplica (I = V·R)', 'Se reduce a la mitad (I = V/R)', 'No cambia (V es el mismo)', 'Se cuadruplica (I ∝ R²)'],1,
    "I=V/R: al doblar R, la corriente cae a la mitad.", id_preg="Corrie_p2_5", peso=2.0)

quiz_opcion_multiple("DEFIENDE TU CONCLUSIÓN: un estudiante afirma que una resistencia 'grande' siempre disipa más potencia porque 'ofrece más oposición'. Conectada a un voltaje fijo, ¿correcto o solo convincente?",
    ['Solo convincente: a voltaje fijo P = V²/R; una R mayor deja pasar menos corriente y disipa menos',
     'Correcto: P = I²R, y en esa fórmula una R mayor siempre da más potencia',
     'Correcto: más oposición significa más choques de los electrones y, por tanto, más calor',
     'Solo convincente: la potencia no depende de R, solo del voltaje aplicado'],
    0,
    'Depende de qué se mantiene fijo. P = I²R vale, pero a voltaje fijo I = V/R también baja con R, y gana el efecto de I²: P = V²/R. A corriente fija (serie) sí disipa más la R mayor.',
    id_preg='Corr_p2_concep_2', peso=2.0)

def _interact_coma(fn, _crudos=(), **controles):
    # Como interact, con lectura de sliders en coma decimal. La salida NO se captura con «with salida:»:
    # se calculan el texto y las figuras y se escriben con append_stdout/append_display_data
    # (evita bloqueos de Voilà en Binder y funciona también en JupyterLite).
    import ipywidgets as _w, io as _io, contextlib as _cl
    import matplotlib.pyplot as _plt
    def _fmt(v, paso, crudo=False):
        # Lectura con coma decimal y 3 cifras significativas (×10ⁿ fuera de 0,001 ≤ |v| < 1000).
        # IntSlider: entero exacto. «crudo»: decimales del paso (slider que representa una medición sin redondear).
        import math
        if isinstance(v, int) and not isinstance(v, bool): return str(v)
        if crudo:
            s = repr(float(paso)); dec = 0 if 'e' in s else len(s.split('.')[1].rstrip('0'))
            return f"{v:.{min(dec, 4)}f}".replace('.', ',').replace('-', '−')
        v = float(v)
        if v == 0 or not math.isfinite(v): return '0' if v == 0 else str(v)
        v = float(f"{v:.2e}")
        if abs(v) >= 1000 or abs(v) < 0.001:
            m, e = f"{v:.2e}".split('e')
            return m.replace('.', ',').replace('-', '−') + '×10' + str(int(e)).translate(str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹'))
        t = f"{v:#.3g}"
        return (t[:-1] if t.endswith('.') else t).replace('.', ',').replace('-', '−')
    filas = []
    for nombre, c in controles.items():
        if isinstance(c, (_w.FloatSlider, _w.IntSlider)):
            c.readout = False
            lab = _w.Label(layout=_w.Layout(min_width='90px'))
            def _act(*_, c=c, lab=lab, cr=(nombre in _crudos)): lab.value = _fmt(c.value, c.step, cr)
            c.observe(_act, 'value'); _act()
            filas.append(_w.HBox([c, lab]))
        else:
            filas.append(c)
    salida = _w.Output()
    def _ejecutar(*_):
        partes, buf = [], _io.StringIO()
        def _vaciar_texto():
            t = buf.getvalue()
            if t: partes.append(('t', t)); buf.seek(0); buf.truncate(0)
        def _capturar(*a, **k):
            _vaciar_texto()
            for num in _plt.get_fignums():
                b = _io.BytesIO(); _plt.figure(num).savefig(b, format='png', bbox_inches='tight')
                partes.append(('i', b.getvalue()))
            _plt.close('all')
        _show = _plt.show; _plt.show = _capturar
        try:
            with _cl.redirect_stdout(buf):
                try: fn(**{k: c.value for k, c in controles.items()})
                except Exception as e: print('⚠ No se pudo calcular con estos valores:', e)
            _capturar()
        finally:
            _plt.show = _show
        _vaciar_texto()
        # Reemplazo en UNA sola asignación: la salida nunca queda vacía (sin saltos de la página)
        import base64 as _b64
        nuevas = []
        for tipo, val in partes:
            if tipo == 't': nuevas.append({'output_type': 'stream', 'name': 'stdout', 'text': val})
            else: nuevas.append({'output_type': 'display_data', 'metadata': {},
                                 'data': {'image/png': _b64.b64encode(val).decode('ascii'), 'text/plain': '<Figure>'}})
        salida.outputs = tuple(nuevas)
    for c in controles.values():
        c.observe(_ejecutar, 'value')
    _ejecutar()
    display(_w.VBox(filas + [salida]))

print("\n--- 🔬 Explora después de responder ---")
print("Los controles NO arrancan en los datos del enunciado. Primero resuelve y responde;\nluego explora tendencias, casos límite y comprueba tu razonamiento.")
_interact_coma(ohm,
    V=FloatSlider(value=180.0,min=10,max=240,step=10,description='V (voltios)'),
    R=FloatSlider(value=110.0,min=10,max=200,step=10,description='R (Ω)'));

---
## Punto 3 — Resistencia de un cable de cobre (Ing. Eléctrica/Transmisión)  ★★★  [PRC, ULC]
Un cable de cobre (ρ = 1,68×10⁻⁸ Ω·m) tiene longitud  L = 100 m  y diámetro  2,00 mm. Se calcula
su resistencia, relevante para pérdidas en líneas.

**Área:**  A = π·r²    **Resistencia:**  R = ρ·L/A

> 🔧 **Aplicación en ingeniería:** en líneas de transmisión y distribución, la resistencia del
> conductor causa pérdidas de energía (P=I²R) y caídas de tensión. Elegir el material (ρ) y el
> calibre (A) correctos minimiza pérdidas: por eso se usan cables gruesos en tramos largos.

> 🤖 **TIP-IA — Traductor de representaciones:**
> Pega la relación R=ρL/A a un asistente: *"Explícame físicamente por qué un cable más LARGO tiene más resistencia pero uno más GRUESO tiene menos. Dame la intuición (analogía con una tubería y el flujo de agua), no solo la fórmula."* Primero intenta explicarlo tú.


In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display

def _3c(x):
    # 3 cifras significativas con coma decimal; ×10ⁿ fuera de 0,001 ≤ |x| < 1000 (textos de gráficas)
    import math
    try: x = float(x)
    except (TypeError, ValueError): return str(x)
    if x == 0 or not math.isfinite(x): return str(x).replace('.', ',') if x else '0'
    x = float(f"{x:.2e}")
    if abs(x) >= 1000 or abs(x) < 0.001:
        e = int(math.floor(math.log10(abs(x)))); m = x / 10**e
        if round(abs(m), 2) >= 10: m /= 10; e += 1
        return f"{m:.2f}".replace('.', ',').replace('-', '−') + '×10' + str(e).translate(str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹'))
    t = f"{x:#.3g}"
    return (t[:-1] if t.endswith('.') else t).replace('.', ',').replace('-', '−')
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

try:
    print_es
except NameError:
    def print_es(*args, **kwargs):
        import re as _re
        print(*[_re.sub(r"(\d)\.(\d)", r"\1,\2", a) if isinstance(a, str) else a for a in args], **kwargs)
# Signo menos tipográfico (−) en todo texto impreso: «-5» → «−5» (no toca guiones entre palabras, rangos ni 1e-3)
import builtins as _bi_br
if not getattr(_bi_br.print, '_menos_br', False):
    def _print_br(*a, _orig=_bi_br.print, **k):
        import re as _re
        _orig(*[_re.sub(r'(?<![\w.,\-])-(?=\d)', '−', x) if isinstance(x, str) else x for x in a], **k)
    _print_br._menos_br = True
    _bi_br.print = _print_br
# --- Cuestionarios Física BR v3: opciones barajadas, crédito por intento, nota sobre el total ---
import random as _rnd_br
try:
    PESOS_BR
except NameError:
    PESOS_BR = {}      # id_preg -> peso de cada pregunta mostrada
try:
    INTENTOS_BR
except NameError:
    INTENTOS_BR = {}   # id_preg -> [intentos, resuelta]
_CREDITO_BR = {1: 1.0, 2: 0.5, 3: 0.25}   # crédito según el intento en que se acierta
_MAX_INT_BR = 3

def _evaluar_intento(id_preg, ok, peso, explicacion, msg_ok, msg_mal):
    est = INTENTOS_BR.setdefault(id_preg, [0, False])
    if est[1]:
        print("✔ Ya resolviste esta pregunta; tu crédito quedó registrado.")
        if explicacion: print_es("→ " + explicacion)
        return
    if est[0] >= _MAX_INT_BR:
        print("⛔ Agotaste los 3 intentos con crédito: esta pregunta ya no suma.")
        if explicacion: print_es("→ " + explicacion)
        return
    est[0] += 1
    n = est[0]
    if ok:
        est[1] = True
        cred = _CREDITO_BR.get(n, 0.0)
        REGISTRO_BR[id_preg] = (cred, float(peso))
        print_es(msg_ok + f"  (intento {n}: crédito {int(cred*100)} %)")
        if explicacion: print_es("→ " + explicacion)
    else:
        REGISTRO_BR.setdefault(id_preg, (0.0, float(peso)))
        if n < _MAX_INT_BR:
            print(msg_mal + f"  Intento {n} de {_MAX_INT_BR}. Revisa tu razonamiento antes de volver a intentar.")
        else:
            print(msg_mal + " Agotaste los 3 intentos: esta pregunta ya no suma. Estudia la explicación:")
            if explicacion: print_es("→ " + explicacion)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    orden = list(range(len(opciones))); _rnd_br.shuffle(orden)   # orden distinto en cada sesión
    correcta = orden.index(indice_correcto)
    letras = "abcdefgh"
    etiquetas = [f"{letras[i]}) {opciones[j]}" for i, j in enumerate(orden)]
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=etiquetas, value=None, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            if radio.index is None:
                print("⚠ Elige una opción."); return
            _evaluar_intento(id_preg, radio.index == correcta, peso, explicacion,
                             "✅ ¡Correcto!", "❌ Incorrecto.")
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.strip().replace(",", "."))
            except ValueError: print("⚠ Escribe un número (usa coma o punto decimal)."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            _evaluar_intento(id_preg, ok, peso, explicacion,
                             f"✅ ¡Correcto!  ({valor_correcto:.3g} {unidad})", "❌ Fuera de rango.")
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not PESOS_BR:
                print("No se encontraron cuestionarios en este taller."); return
            peso_total = sum(PESOS_BR.values())
            peso_ok = sum(float(c)*p for c, p in REGISTRO_BR.values())
            nota = 5.0*peso_ok/peso_total if peso_total > 0 else 0.0
            total = len(PESOS_BR)
            resp = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0])[0] > 0)
            prim = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0, False])[1] and INTENTOS_BR[k][0] == 1)
            print("="*46)
            print("          CALIFICACIÓN FORMATIVA")
            print("="*46)
            print(f"  Preguntas respondidas:       {resp} de {total}")
            print(f"  Acertadas al primer intento: {prim}")
            print_es(f"  Puntaje ponderado:           {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*46)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*46)
            if resp < total: print(f"  ⚠ Faltan {total-resp} preguntas: cuentan como 0.")
            if nota >= 3.5: print("  ✅ ¡Buen trabajo! Dominas los conceptos.")
            elif nota >= 3.0: print("  🟡 Vas bien, repasa lo que fallaste.")
            else: print("  🔴 Repasa la teoría y vuelve a los puntos.")
            print("\n  Crédito por pregunta: 100 % al primer intento,")
            print("  50 % al segundo, 25 % al tercero. Predice y")
            print("  razona ANTES de verificar: adivinar cuesta nota.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>📊 Tu calificación del taller</h3>"
        "<i>La nota se calcula sobre TODAS las preguntas del taller y premia acertar al primer intento.</i>"),
        boton, salida]))


def esquema():
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(9,4))
    # IZQUIERDA: más largo = más resistencia
    ax1.add_patch(plt.Rectangle((0.5,2.5),2,0.6,color='#b87333',alpha=0.8))
    ax1.add_patch(plt.Rectangle((0.5,1),5,0.6,color='#b87333',alpha=0.8))
    ax1.text(1.5,3.3,'corto: R baja',ha='center',fontsize=9,color='#27ae60',fontweight='bold')
    ax1.text(3,0.5,'largo: R alta',ha='center',fontsize=9,color='#c0392b',fontweight='bold')
    ax1.text(3,2,'R ∝ longitud',ha='center',fontsize=9,color='#2c3e50')
    ax1.set_xlim(0,6); ax1.set_ylim(0,3.7); ax1.axis('off')
    ax1.set_title('Más largo → más resistencia',fontsize=10)
    # DERECHA: más grueso = menos resistencia
    ax2.add_patch(plt.Rectangle((0.5,2.5),4,0.4,color='#b87333',alpha=0.8))
    ax2.add_patch(plt.Rectangle((0.5,0.7),4,1.1,color='#b87333',alpha=0.8))
    ax2.text(2.5,3.1,'delgado: R alta',ha='center',fontsize=9,color='#c0392b',fontweight='bold')
    ax2.text(2.5,0.3,'grueso: R baja',ha='center',fontsize=9,color='#27ae60',fontweight='bold')
    ax2.text(2.5,2,'R ∝ 1/área',ha='center',fontsize=9,color='#2c3e50')
    ax2.set_xlim(0,5); ax2.set_ylim(0,3.5); ax2.axis('off')
    ax2.set_title('Más grueso → menos resistencia',fontsize=10)
    plt.suptitle('Resistencia según la geometría: R = ρ·L/A',fontsize=11,fontweight='bold',color='#2c3e50')
    plt.tight_layout(); plt.show()
esquema()

def resistividad(L=100.0, d_mm=2.0, rho_n=1.68):
    rho=rho_n*1e-8; A=np.pi*(d_mm/2000)**2; R=rho*L/A
    Ls=np.linspace(10,300,100); Rs=rho*Ls/A
    plt.figure(figsize=(9,4.2)); plt.plot(Ls,Rs,lw=2,color='purple')
    plt.scatter([L],[R],color='red',s=60,zorder=5,label=f'L={L:.0f} m → R={_3c(R)} Ω')
    plt.xlabel('longitud L (m)'); plt.ylabel('resistencia R (Ω)')
    plt.title(f'R=ρL/A  ·  A={_3c(A*1e6)} mm²'); plt.legend(); plt.grid(alpha=.3)
    plt.tight_layout(); plt.show()


def solucion():
    import re as _re
    def _pc(*a,**k):
        def _f(t):
            if not isinstance(t,str): return t
            _sup='⁰¹²³⁴⁵⁶⁷⁸⁹'
            def _sci(m):
                mant=m.group(1); exp=int(m.group(2))
                e=('⁻' if exp<0 else '')+''.join(_sup[int(d)] for d in str(abs(exp)))
                return mant+'×10'+e
            t=_re.sub(r'(\d+(?:\.\d+)?)[eE]([+-]?\d+)', _sci, t)
            _u=r'(?:s|km/s|km|m/s²|m/s|m|N·m|N|J|W|A|V|T|Ω|C|F|Hz|rad/s²|rad/s|rad|kg·m²/s|kg·m²|kg|Pa|kPa|µF|nC|µC|pF|mWb|Wb|µT|mT|kW|MW|mA|kΩ|MΩ|rpm|°)'
            def _c3(m):
                num=m.group(2); n=int(num)
                if n==0: return m.group(0)
                if len(num)==1: nv=num+'.00'
                elif len(num)==2: nv=num+'.0'
                else: nv=num
                return m.group(1)+nv+m.group(3)
            t=_re.sub(r'(?<![\d.,])([\s=(×])(\d{1,3})(\s'+_u+r'\b)', _c3, t)
            t=_re.sub(r'(\d)\.(\d)', r'\1,\2', t)
            return t
        print(*[_f(x) for x in a], **k)

    rho=1.68e-8; L=100.0; A=np.pi*(2.0/2000)**2; R=rho*L/A
    _pc(f"Área: A = πr² = π·(1,00e-3)² = {A:.3g} m² = {A*1e6:.3g} mm²")
    _pc(f"Resistencia: R = ρL/A = 1,68e-8·100/{A:.2e} = {R:.3g} Ω")
    _pc("A mayor longitud, mayor R; a mayor área (cable más grueso), menor R.")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 3 ---")
quiz_numerico("Un cable de cobre tiene un diámetro de 2,00 mm. Encuentre el área de su sección transversal.",3.14,0.05,"mm²","A=πr²≈3,14 mm².", id_preg="Corrie_p3_6", peso=3.0)
quiz_opcion_multiple("La resistencia de un cable es R=ρL/A. ¿Por qué los cables de alta corriente (como los de una subestación) son GRUESOS?",['Por estética y para facilitar la instalación', 'Más área A reduce R y las pérdidas de energía', 'Porque pesan más y resisten mejor el viento', 'Para que duren más años sin oxidarse'],1,"R=ρL/A: a mayor área, menor resistencia. Un cable grueso reduce la resistencia y con ella las pérdidas por calor (I²R). Por eso las líneas de alta corriente usan conductores de gran sección.", id_preg="Corrie_p3_7", peso=3.0)
quiz_opcion_multiple("Para reducir la resistencia de una línea de transmisión larga, ¿qué conviene hacer con el conductor?",
    ['Cable más delgado (menos material)', 'Cable más grueso (mayor área)', 'Cable más largo (más camino)', 'Bajar el voltaje de la línea'],1,
    "R=ρL/A: mayor área ⟹ menor resistencia. Por eso se usan cables gruesos.", id_preg="Corrie_p3_8", peso=3.0)

quiz_opcion_multiple("DEFIENDE TU CONCLUSIÓN: un técnico dice que para reducir pérdidas en una línea larga conviene un cable MÁS DELGADO porque 'pesa menos y es más barato'. ¿Correcto o solo convincente?",
    ['Solo convincente: R = ρL/A; más delgado es más R, más pérdidas I²R y más caída de tensión',
     'Correcto: un cable más liviano transporta la corriente con menos esfuerzo y, por eso, pierde menos energía a lo largo de la línea',
     'Correcto: las pérdidas dependen solo de la longitud de la línea, no del calibre del cable',
     'Solo convincente: el calibre no influye en las pérdidas, solo en el precio del cable'],
    0,
    'El ahorro en cobre es real, pero se paga cada hora en energía disipada y calentamiento (RETIE limita la caída de tensión). Las pérdidas dependen de L y también de A.',
    id_preg='Corr_p3_concep_3', peso=2.0)

def _interact_coma(fn, _crudos=(), **controles):
    # Como interact, con lectura de sliders en coma decimal. La salida NO se captura con «with salida:»:
    # se calculan el texto y las figuras y se escriben con append_stdout/append_display_data
    # (evita bloqueos de Voilà en Binder y funciona también en JupyterLite).
    import ipywidgets as _w, io as _io, contextlib as _cl
    import matplotlib.pyplot as _plt
    def _fmt(v, paso, crudo=False):
        # Lectura con coma decimal y 3 cifras significativas (×10ⁿ fuera de 0,001 ≤ |v| < 1000).
        # IntSlider: entero exacto. «crudo»: decimales del paso (slider que representa una medición sin redondear).
        import math
        if isinstance(v, int) and not isinstance(v, bool): return str(v)
        if crudo:
            s = repr(float(paso)); dec = 0 if 'e' in s else len(s.split('.')[1].rstrip('0'))
            return f"{v:.{min(dec, 4)}f}".replace('.', ',').replace('-', '−')
        v = float(v)
        if v == 0 or not math.isfinite(v): return '0' if v == 0 else str(v)
        v = float(f"{v:.2e}")
        if abs(v) >= 1000 or abs(v) < 0.001:
            m, e = f"{v:.2e}".split('e')
            return m.replace('.', ',').replace('-', '−') + '×10' + str(int(e)).translate(str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹'))
        t = f"{v:#.3g}"
        return (t[:-1] if t.endswith('.') else t).replace('.', ',').replace('-', '−')
    filas = []
    for nombre, c in controles.items():
        if isinstance(c, (_w.FloatSlider, _w.IntSlider)):
            c.readout = False
            lab = _w.Label(layout=_w.Layout(min_width='90px'))
            def _act(*_, c=c, lab=lab, cr=(nombre in _crudos)): lab.value = _fmt(c.value, c.step, cr)
            c.observe(_act, 'value'); _act()
            filas.append(_w.HBox([c, lab]))
        else:
            filas.append(c)
    salida = _w.Output()
    def _ejecutar(*_):
        partes, buf = [], _io.StringIO()
        def _vaciar_texto():
            t = buf.getvalue()
            if t: partes.append(('t', t)); buf.seek(0); buf.truncate(0)
        def _capturar(*a, **k):
            _vaciar_texto()
            for num in _plt.get_fignums():
                b = _io.BytesIO(); _plt.figure(num).savefig(b, format='png', bbox_inches='tight')
                partes.append(('i', b.getvalue()))
            _plt.close('all')
        _show = _plt.show; _plt.show = _capturar
        try:
            with _cl.redirect_stdout(buf):
                try: fn(**{k: c.value for k, c in controles.items()})
                except Exception as e: print('⚠ No se pudo calcular con estos valores:', e)
            _capturar()
        finally:
            _plt.show = _show
        _vaciar_texto()
        # Reemplazo en UNA sola asignación: la salida nunca queda vacía (sin saltos de la página)
        import base64 as _b64
        nuevas = []
        for tipo, val in partes:
            if tipo == 't': nuevas.append({'output_type': 'stream', 'name': 'stdout', 'text': val})
            else: nuevas.append({'output_type': 'display_data', 'metadata': {},
                                 'data': {'image/png': _b64.b64encode(val).decode('ascii'), 'text/plain': '<Figure>'}})
        salida.outputs = tuple(nuevas)
    for c in controles.values():
        c.observe(_ejecutar, 'value')
    _ejecutar()
    display(_w.VBox(filas + [salida]))

def _explora_sin_cifras(fn):
    # Opción 3: exploración cualitativa; las cifras de este punto solo están en «Mostrar solución».
    import functools, io, contextlib, re as _re
    _pat = _re.compile(r'(?<![\w.,/])[-−]?\d+(?:[.,]\d+)?(?:\s?×\s?10[⁻⁰¹²³⁴⁵⁶⁷⁸⁹]+|[eE][+-]?\d+)?(?![\wπ·(]|[.,]\d)')
    def _m(t): return _pat.sub('…', t)
    def _ocultar():
        for num in plt.get_fignums():
            fig = plt.figure(num)
            textos = list(fig.texts) + ([fig._suptitle] if getattr(fig, '_suptitle', None) else [])
            for ax in fig.axes:
                textos += [ax.title, ax._left_title, ax._right_title] + list(ax.texts)
                if ax.get_legend(): textos += list(ax.get_legend().get_texts())
            for t in textos: t.set_text(_m(t.get_text()))
    @functools.wraps(fn)
    def envuelta(*a, **k):
        _show = plt.show
        def _show2(*sa, **sk):
            _ocultar(); _show(*sa, **sk)
        buf = io.StringIO(); plt.show = _show2
        try:
            with contextlib.redirect_stdout(buf): fn(*a, **k)
            _ocultar()
        finally:
            plt.show = _show
        for linea in buf.getvalue().splitlines():
            print(linea if ('⚠' in linea or '⛔' in linea) else _m(linea))
        print("(Exploración cualitativa: observa tendencias. Las cifras de este punto están en «Mostrar solución».)")
    return envuelta

print("\n--- 🔬 Explora después de responder ---")
print("Los controles NO arrancan en los datos del enunciado. Primero resuelve y responde;\nluego explora tendencias, casos límite y comprueba tu razonamiento.")
_interact_coma(_explora_sin_cifras(resistividad),
    L=FloatSlider(value=170.0,min=10,max=300,step=10,description='L (m)'),
    d_mm=FloatSlider(value=3.0,min=0.5,max=5,step=0.5,description='diám (mm)'),
    rho_n=FloatSlider(value=4.5,min=1.0,max=12,step=0.5,description='ρ (×10⁻⁸)'));

---
## Punto 4 — Potencia disipada y costo energético (Ing. Eléctrica/Industrial)  ★★★  [PRC, UCM]
Una resistencia de calefacción industrial  R = 10,0 Ω  conduce una corriente  I = 5,00 A  durante
t = 3,00 h. Se calcula la potencia, la energía consumida y su costo.

**Potencia:**  P = I²·R    **Energía:**  E = P·t    **Costo:**  E(kWh) × tarifa

> 🔧 **Aplicación en ingeniería:** dimensionar equipos y estimar el costo operativo parte de la
> potencia disipada. En la industria, la energía se factura en kWh; calcular consumo y costo permite
> evaluar la eficiencia de procesos y justificar inversiones en equipos de menor consumo.

> 🤖 **TIP-IA — Generador de problemas:**
> Pídele: *"Créame una variante de este problema con otra resistencia, corriente y tarifa eléctrica, y pídeme calcular la potencia, la energía en kWh y el costo. No me des la solución hasta que yo la intente."* Primero resuelves tú, después verificas.


In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display

def _3c(x):
    # 3 cifras significativas con coma decimal; ×10ⁿ fuera de 0,001 ≤ |x| < 1000 (textos de gráficas)
    import math
    try: x = float(x)
    except (TypeError, ValueError): return str(x)
    if x == 0 or not math.isfinite(x): return str(x).replace('.', ',') if x else '0'
    x = float(f"{x:.2e}")
    if abs(x) >= 1000 or abs(x) < 0.001:
        e = int(math.floor(math.log10(abs(x)))); m = x / 10**e
        if round(abs(m), 2) >= 10: m /= 10; e += 1
        return f"{m:.2f}".replace('.', ',').replace('-', '−') + '×10' + str(e).translate(str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹'))
    t = f"{x:#.3g}"
    return (t[:-1] if t.endswith('.') else t).replace('.', ',').replace('-', '−')
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

try:
    print_es
except NameError:
    def print_es(*args, **kwargs):
        import re as _re
        print(*[_re.sub(r"(\d)\.(\d)", r"\1,\2", a) if isinstance(a, str) else a for a in args], **kwargs)
# Signo menos tipográfico (−) en todo texto impreso: «-5» → «−5» (no toca guiones entre palabras, rangos ni 1e-3)
import builtins as _bi_br
if not getattr(_bi_br.print, '_menos_br', False):
    def _print_br(*a, _orig=_bi_br.print, **k):
        import re as _re
        _orig(*[_re.sub(r'(?<![\w.,\-])-(?=\d)', '−', x) if isinstance(x, str) else x for x in a], **k)
    _print_br._menos_br = True
    _bi_br.print = _print_br
# --- Cuestionarios Física BR v3: opciones barajadas, crédito por intento, nota sobre el total ---
import random as _rnd_br
try:
    PESOS_BR
except NameError:
    PESOS_BR = {}      # id_preg -> peso de cada pregunta mostrada
try:
    INTENTOS_BR
except NameError:
    INTENTOS_BR = {}   # id_preg -> [intentos, resuelta]
_CREDITO_BR = {1: 1.0, 2: 0.5, 3: 0.25}   # crédito según el intento en que se acierta
_MAX_INT_BR = 3

def _evaluar_intento(id_preg, ok, peso, explicacion, msg_ok, msg_mal):
    est = INTENTOS_BR.setdefault(id_preg, [0, False])
    if est[1]:
        print("✔ Ya resolviste esta pregunta; tu crédito quedó registrado.")
        if explicacion: print_es("→ " + explicacion)
        return
    if est[0] >= _MAX_INT_BR:
        print("⛔ Agotaste los 3 intentos con crédito: esta pregunta ya no suma.")
        if explicacion: print_es("→ " + explicacion)
        return
    est[0] += 1
    n = est[0]
    if ok:
        est[1] = True
        cred = _CREDITO_BR.get(n, 0.0)
        REGISTRO_BR[id_preg] = (cred, float(peso))
        print_es(msg_ok + f"  (intento {n}: crédito {int(cred*100)} %)")
        if explicacion: print_es("→ " + explicacion)
    else:
        REGISTRO_BR.setdefault(id_preg, (0.0, float(peso)))
        if n < _MAX_INT_BR:
            print(msg_mal + f"  Intento {n} de {_MAX_INT_BR}. Revisa tu razonamiento antes de volver a intentar.")
        else:
            print(msg_mal + " Agotaste los 3 intentos: esta pregunta ya no suma. Estudia la explicación:")
            if explicacion: print_es("→ " + explicacion)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    orden = list(range(len(opciones))); _rnd_br.shuffle(orden)   # orden distinto en cada sesión
    correcta = orden.index(indice_correcto)
    letras = "abcdefgh"
    etiquetas = [f"{letras[i]}) {opciones[j]}" for i, j in enumerate(orden)]
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=etiquetas, value=None, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            if radio.index is None:
                print("⚠ Elige una opción."); return
            _evaluar_intento(id_preg, radio.index == correcta, peso, explicacion,
                             "✅ ¡Correcto!", "❌ Incorrecto.")
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.strip().replace(",", "."))
            except ValueError: print("⚠ Escribe un número (usa coma o punto decimal)."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            _evaluar_intento(id_preg, ok, peso, explicacion,
                             f"✅ ¡Correcto!  ({valor_correcto:.3g} {unidad})", "❌ Fuera de rango.")
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not PESOS_BR:
                print("No se encontraron cuestionarios en este taller."); return
            peso_total = sum(PESOS_BR.values())
            peso_ok = sum(float(c)*p for c, p in REGISTRO_BR.values())
            nota = 5.0*peso_ok/peso_total if peso_total > 0 else 0.0
            total = len(PESOS_BR)
            resp = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0])[0] > 0)
            prim = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0, False])[1] and INTENTOS_BR[k][0] == 1)
            print("="*46)
            print("          CALIFICACIÓN FORMATIVA")
            print("="*46)
            print(f"  Preguntas respondidas:       {resp} de {total}")
            print(f"  Acertadas al primer intento: {prim}")
            print_es(f"  Puntaje ponderado:           {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*46)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*46)
            if resp < total: print(f"  ⚠ Faltan {total-resp} preguntas: cuentan como 0.")
            if nota >= 3.5: print("  ✅ ¡Buen trabajo! Dominas los conceptos.")
            elif nota >= 3.0: print("  🟡 Vas bien, repasa lo que fallaste.")
            else: print("  🔴 Repasa la teoría y vuelve a los puntos.")
            print("\n  Crédito por pregunta: 100 % al primer intento,")
            print("  50 % al segundo, 25 % al tercero. Predice y")
            print("  razona ANTES de verificar: adivinar cuesta nota.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>📊 Tu calificación del taller</h3>"
        "<i>La nota se calcula sobre TODAS las preguntas del taller y premia acertar al primer intento.</i>"),
        boton, salida]))


def esquema():
    fig,ax=plt.subplots(figsize=(7,4))
    # resistencia que se calienta
    xr=np.linspace(2,6,30); yr=2+0.2*np.sin(np.linspace(0,8*np.pi,30))
    ax.plot(xr,yr,color='#e67e22',lw=3)
    ax.text(4,1.4,'resistencia',ha='center',fontsize=9,color='#e67e22',fontweight='bold')
    # ondas de calor saliendo
    for x in [3,4,5]:
        for i,y in enumerate([2.6,3.1,3.6]):
            ax.plot([x-0.15,x,x+0.15],[y,y+0.15,y],color='#c0392b',lw=1.2,alpha=0.7-i*0.15)
    ax.text(4,4,'calor disipado',ha='center',fontsize=9,color='#c0392b',fontweight='bold')
    # corriente entrando y saliendo
    ax.annotate('',xy=(2,2),xytext=(0.5,2),arrowprops=dict(arrowstyle='-|>',color='#27ae60',lw=2))
    ax.annotate('',xy=(7.5,2),xytext=(6,2),arrowprops=dict(arrowstyle='-|>',color='#27ae60',lw=2))
    ax.text(1,2.3,'I',fontsize=11,color='#27ae60',fontweight='bold')
    ax.text(4,0.5,'La corriente al pasar por R disipa potencia: P = I²·R = V·I',
            ha='center',fontsize=9,color='#2c3e50',fontweight='bold')
    ax.set_xlim(0,8); ax.set_ylim(0,4.5); ax.axis('off')
    ax.set_title('Potencia disipada: energía eléctrica → calor')
    plt.tight_layout(); plt.show()
esquema()

def potencia_costo(R=10.0, I=5.0, t_h=3.0, tarifa=800.0):
    P=I**2*R; E_kwh=P*t_h/1000; costo=E_kwh*tarifa
    Is=np.linspace(1,10,100); Ps=Is**2*R
    plt.figure(figsize=(9,4.2)); plt.plot(Is,Ps,lw=2,color='crimson')
    plt.scatter([I],[P],color='blue',s=60,zorder=5,label=f'I={I:.0f} A → P={P:.0f} W')
    plt.xlabel('corriente I (A)'); plt.ylabel('potencia P (W)')
    plt.title(f'P=I²R  ·  E={_3c(E_kwh)} kWh  ·  costo=${costo:.0f}'); plt.legend(); plt.grid(alpha=.3)
    plt.tight_layout(); plt.show()


def solucion():
    import re as _re
    def _pc(*a,**k):
        def _f(t):
            if not isinstance(t,str): return t
            _sup='⁰¹²³⁴⁵⁶⁷⁸⁹'
            def _sci(m):
                mant=m.group(1); exp=int(m.group(2))
                e=('⁻' if exp<0 else '')+''.join(_sup[int(d)] for d in str(abs(exp)))
                return mant+'×10'+e
            t=_re.sub(r'(\d+(?:\.\d+)?)[eE]([+-]?\d+)', _sci, t)
            _u=r'(?:s|km/s|km|m/s²|m/s|m|N·m|N|J|W|A|V|T|Ω|C|F|Hz|rad/s²|rad/s|rad|kg·m²/s|kg·m²|kg|Pa|kPa|µF|nC|µC|pF|mWb|Wb|µT|mT|kW|MW|mA|kΩ|MΩ|rpm|°)'
            def _c3(m):
                num=m.group(2); n=int(num)
                if n==0: return m.group(0)
                if len(num)==1: nv=num+'.00'
                elif len(num)==2: nv=num+'.0'
                else: nv=num
                return m.group(1)+nv+m.group(3)
            t=_re.sub(r'(?<![\d.,])([\s=(×])(\d{1,3})(\s'+_u+r'\b)', _c3, t)
            t=_re.sub(r'(\d)\.(\d)', r'\1,\2', t)
            return t
        print(*[_f(x) for x in a], **k)

    R=10.0; I=5.0; P=I**2*R; t=3.0; E=P*t/1000
    _pc(f"Potencia: P = I²R = 5,00²·10,0 = {P:.3g} W")
    _pc(f"Energía en 3,00 h: E = P·t = 250·3,00 = {P*t:.0f} Wh = {E:.3g} kWh")
    _pc(f"Costo (a $800/kWh): {E:.2f}·800 = ${E*800:.0f}")
    _pc("La potencia crece con el CUADRADO de la corriente (P=I²R).")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 4 ---")
quiz_numerico("Una resistencia de calefacción de 10,0 Ω conduce una corriente de 5,00 A. Encuentre la potencia que disipa.",250,0.04,"W","P=I²R=250 W.", id_preg="Corrie_p4_9", peso=3.0)
quiz_opcion_multiple("Una resistencia disipa potencia según P=I²R. Si la corriente se duplica, ¿cómo cambia el calor disipado?",['Se duplica (P ∝ I)', 'Se cuadruplica (P ∝ I²)', 'No cambia (R es fija)', 'Se reduce (P ∝ 1/I)'],1,"La potencia disipada va con el CUADRADO de la corriente (P=I²R). Al duplicar I, el calor se cuadruplica. Por eso las sobrecargas de corriente son tan peligrosas: calientan desproporcionadamente.", id_preg="Corrie_p4_10", peso=3.0)
quiz_opcion_multiple("En una resistencia, si se duplica la corriente que circula, ¿qué ocurre con la potencia disipada?",
    ['Se duplica (P ∝ I)', 'Se cuadruplica (P ∝ I²)', 'No cambia (R es fija)', 'Se reduce (P ∝ 1/I)'],1,
    "P=I²R: al doblar I, la potencia crece 4 veces.", id_preg="Corrie_p4_11", peso=3.0)

quiz_opcion_multiple('DEFIENDE TU CONCLUSIÓN: un estudiante dice que duplicar la corriente en el mismo calefactor cuadruplica la potencia y el costo por hora. ¿Correcto o solo convincente?',
    ['Correcto: con R fija, P = I²R; con 2I la potencia es 4P, y el costo por hora también',
     'Correcto: se cuadruplica porque al duplicar la corriente la resistencia también se duplica',
     'Solo convincente: P = V·I, así que con el doble de corriente la potencia solo se duplica',
     'Solo convincente: el costo depende de la energía, no de la corriente, así que no cambia'],
    0,
    'Esta vez es CORRECTO (con R constante). La opción «P = V·I, así que con…» olvida que, con R fija, duplicar I exige duplicar V: P = (2V)(2I) = 4P. El costo sí depende de la energía, que es P·t.',
    id_preg='Corr_p4_concep_4', peso=2.0)

def _interact_coma(fn, _crudos=(), **controles):
    # Como interact, con lectura de sliders en coma decimal. La salida NO se captura con «with salida:»:
    # se calculan el texto y las figuras y se escriben con append_stdout/append_display_data
    # (evita bloqueos de Voilà en Binder y funciona también en JupyterLite).
    import ipywidgets as _w, io as _io, contextlib as _cl
    import matplotlib.pyplot as _plt
    def _fmt(v, paso, crudo=False):
        # Lectura con coma decimal y 3 cifras significativas (×10ⁿ fuera de 0,001 ≤ |v| < 1000).
        # IntSlider: entero exacto. «crudo»: decimales del paso (slider que representa una medición sin redondear).
        import math
        if isinstance(v, int) and not isinstance(v, bool): return str(v)
        if crudo:
            s = repr(float(paso)); dec = 0 if 'e' in s else len(s.split('.')[1].rstrip('0'))
            return f"{v:.{min(dec, 4)}f}".replace('.', ',').replace('-', '−')
        v = float(v)
        if v == 0 or not math.isfinite(v): return '0' if v == 0 else str(v)
        v = float(f"{v:.2e}")
        if abs(v) >= 1000 or abs(v) < 0.001:
            m, e = f"{v:.2e}".split('e')
            return m.replace('.', ',').replace('-', '−') + '×10' + str(int(e)).translate(str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹'))
        t = f"{v:#.3g}"
        return (t[:-1] if t.endswith('.') else t).replace('.', ',').replace('-', '−')
    filas = []
    for nombre, c in controles.items():
        if isinstance(c, (_w.FloatSlider, _w.IntSlider)):
            c.readout = False
            lab = _w.Label(layout=_w.Layout(min_width='90px'))
            def _act(*_, c=c, lab=lab, cr=(nombre in _crudos)): lab.value = _fmt(c.value, c.step, cr)
            c.observe(_act, 'value'); _act()
            filas.append(_w.HBox([c, lab]))
        else:
            filas.append(c)
    salida = _w.Output()
    def _ejecutar(*_):
        partes, buf = [], _io.StringIO()
        def _vaciar_texto():
            t = buf.getvalue()
            if t: partes.append(('t', t)); buf.seek(0); buf.truncate(0)
        def _capturar(*a, **k):
            _vaciar_texto()
            for num in _plt.get_fignums():
                b = _io.BytesIO(); _plt.figure(num).savefig(b, format='png', bbox_inches='tight')
                partes.append(('i', b.getvalue()))
            _plt.close('all')
        _show = _plt.show; _plt.show = _capturar
        try:
            with _cl.redirect_stdout(buf):
                try: fn(**{k: c.value for k, c in controles.items()})
                except Exception as e: print('⚠ No se pudo calcular con estos valores:', e)
            _capturar()
        finally:
            _plt.show = _show
        _vaciar_texto()
        # Reemplazo en UNA sola asignación: la salida nunca queda vacía (sin saltos de la página)
        import base64 as _b64
        nuevas = []
        for tipo, val in partes:
            if tipo == 't': nuevas.append({'output_type': 'stream', 'name': 'stdout', 'text': val})
            else: nuevas.append({'output_type': 'display_data', 'metadata': {},
                                 'data': {'image/png': _b64.b64encode(val).decode('ascii'), 'text/plain': '<Figure>'}})
        salida.outputs = tuple(nuevas)
    for c in controles.values():
        c.observe(_ejecutar, 'value')
    _ejecutar()
    display(_w.VBox(filas + [salida]))

def _explora_sin_cifras(fn):
    # Opción 3: exploración cualitativa; las cifras de este punto solo están en «Mostrar solución».
    import functools, io, contextlib, re as _re
    _pat = _re.compile(r'(?<![\w.,/])[-−]?\d+(?:[.,]\d+)?(?:\s?×\s?10[⁻⁰¹²³⁴⁵⁶⁷⁸⁹]+|[eE][+-]?\d+)?(?![\wπ·(]|[.,]\d)')
    def _m(t): return _pat.sub('…', t)
    def _ocultar():
        for num in plt.get_fignums():
            fig = plt.figure(num)
            textos = list(fig.texts) + ([fig._suptitle] if getattr(fig, '_suptitle', None) else [])
            for ax in fig.axes:
                textos += [ax.title, ax._left_title, ax._right_title] + list(ax.texts)
                if ax.get_legend(): textos += list(ax.get_legend().get_texts())
            for t in textos: t.set_text(_m(t.get_text()))
    @functools.wraps(fn)
    def envuelta(*a, **k):
        _show = plt.show
        def _show2(*sa, **sk):
            _ocultar(); _show(*sa, **sk)
        buf = io.StringIO(); plt.show = _show2
        try:
            with contextlib.redirect_stdout(buf): fn(*a, **k)
            _ocultar()
        finally:
            plt.show = _show
        for linea in buf.getvalue().splitlines():
            print(linea if ('⚠' in linea or '⛔' in linea) else _m(linea))
        print("(Exploración cualitativa: observa tendencias. Las cifras de este punto están en «Mostrar solución».)")
    return envuelta

print("\n--- 🔬 Explora después de responder ---")
print("Los controles NO arrancan en los datos del enunciado. Primero resuelve y responde;\nluego explora tendencias, casos límite y comprueba tu razonamiento.")
_interact_coma(_explora_sin_cifras(potencia_costo),
    R=FloatSlider(value=22.0,min=2,max=50,step=2,description='R (Ω)'),
    I=FloatSlider(value=7.0,min=1,max=10,step=0.5,description='I (A)'),
    t_h=FloatSlider(value=6.0,min=1,max=12,step=1,description='t (h)'),
    tarifa=FloatSlider(value=1000.0,min=400,max=1200,step=50,description='$/kWh'));

---
## Punto 5 — Variación de la resistencia con la temperatura (Ing. Electrónica/Instrumentación)  ★★★  [PRC, ULC]
Una resistencia de  R₀ = 100 Ω  (a 20°C) de coeficiente  α = 0,00390 /°C  se calienta  ΔT = 50,0°C.
Se calcula su nueva resistencia.

**Variación con temperatura:**  R(T) = R₀·(1 + α·ΔT)

> 🔧 **Aplicación en ingeniería:** los sensores de temperatura tipo RTD (como el Pt100) se basan en
> este efecto: miden temperatura a partir del cambio de resistencia. En instrumentación industrial,
> conocer α permite convertir una lectura de resistencia en una temperatura precisa.

> 🤖 **TIP-IA — Abogado del diablo:**
> Dile a un asistente: *"Voy a sostener que la resistencia de un metal NO cambia con la temperatura, que es una propiedad fija del material. Dame el mejor contraargumento y un ejemplo real."* (Pista: es justo el principio del sensor Pt100). Primero razona tú.


In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display

def _3c(x):
    # 3 cifras significativas con coma decimal; ×10ⁿ fuera de 0,001 ≤ |x| < 1000 (textos de gráficas)
    import math
    try: x = float(x)
    except (TypeError, ValueError): return str(x)
    if x == 0 or not math.isfinite(x): return str(x).replace('.', ',') if x else '0'
    x = float(f"{x:.2e}")
    if abs(x) >= 1000 or abs(x) < 0.001:
        e = int(math.floor(math.log10(abs(x)))); m = x / 10**e
        if round(abs(m), 2) >= 10: m /= 10; e += 1
        return f"{m:.2f}".replace('.', ',').replace('-', '−') + '×10' + str(e).translate(str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹'))
    t = f"{x:#.3g}"
    return (t[:-1] if t.endswith('.') else t).replace('.', ',').replace('-', '−')
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

try:
    print_es
except NameError:
    def print_es(*args, **kwargs):
        import re as _re
        print(*[_re.sub(r"(\d)\.(\d)", r"\1,\2", a) if isinstance(a, str) else a for a in args], **kwargs)
# Signo menos tipográfico (−) en todo texto impreso: «-5» → «−5» (no toca guiones entre palabras, rangos ni 1e-3)
import builtins as _bi_br
if not getattr(_bi_br.print, '_menos_br', False):
    def _print_br(*a, _orig=_bi_br.print, **k):
        import re as _re
        _orig(*[_re.sub(r'(?<![\w.,\-])-(?=\d)', '−', x) if isinstance(x, str) else x for x in a], **k)
    _print_br._menos_br = True
    _bi_br.print = _print_br
# --- Cuestionarios Física BR v3: opciones barajadas, crédito por intento, nota sobre el total ---
import random as _rnd_br
try:
    PESOS_BR
except NameError:
    PESOS_BR = {}      # id_preg -> peso de cada pregunta mostrada
try:
    INTENTOS_BR
except NameError:
    INTENTOS_BR = {}   # id_preg -> [intentos, resuelta]
_CREDITO_BR = {1: 1.0, 2: 0.5, 3: 0.25}   # crédito según el intento en que se acierta
_MAX_INT_BR = 3

def _evaluar_intento(id_preg, ok, peso, explicacion, msg_ok, msg_mal):
    est = INTENTOS_BR.setdefault(id_preg, [0, False])
    if est[1]:
        print("✔ Ya resolviste esta pregunta; tu crédito quedó registrado.")
        if explicacion: print_es("→ " + explicacion)
        return
    if est[0] >= _MAX_INT_BR:
        print("⛔ Agotaste los 3 intentos con crédito: esta pregunta ya no suma.")
        if explicacion: print_es("→ " + explicacion)
        return
    est[0] += 1
    n = est[0]
    if ok:
        est[1] = True
        cred = _CREDITO_BR.get(n, 0.0)
        REGISTRO_BR[id_preg] = (cred, float(peso))
        print_es(msg_ok + f"  (intento {n}: crédito {int(cred*100)} %)")
        if explicacion: print_es("→ " + explicacion)
    else:
        REGISTRO_BR.setdefault(id_preg, (0.0, float(peso)))
        if n < _MAX_INT_BR:
            print(msg_mal + f"  Intento {n} de {_MAX_INT_BR}. Revisa tu razonamiento antes de volver a intentar.")
        else:
            print(msg_mal + " Agotaste los 3 intentos: esta pregunta ya no suma. Estudia la explicación:")
            if explicacion: print_es("→ " + explicacion)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    orden = list(range(len(opciones))); _rnd_br.shuffle(orden)   # orden distinto en cada sesión
    correcta = orden.index(indice_correcto)
    letras = "abcdefgh"
    etiquetas = [f"{letras[i]}) {opciones[j]}" for i, j in enumerate(orden)]
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=etiquetas, value=None, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            if radio.index is None:
                print("⚠ Elige una opción."); return
            _evaluar_intento(id_preg, radio.index == correcta, peso, explicacion,
                             "✅ ¡Correcto!", "❌ Incorrecto.")
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.strip().replace(",", "."))
            except ValueError: print("⚠ Escribe un número (usa coma o punto decimal)."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            _evaluar_intento(id_preg, ok, peso, explicacion,
                             f"✅ ¡Correcto!  ({valor_correcto:.3g} {unidad})", "❌ Fuera de rango.")
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not PESOS_BR:
                print("No se encontraron cuestionarios en este taller."); return
            peso_total = sum(PESOS_BR.values())
            peso_ok = sum(float(c)*p for c, p in REGISTRO_BR.values())
            nota = 5.0*peso_ok/peso_total if peso_total > 0 else 0.0
            total = len(PESOS_BR)
            resp = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0])[0] > 0)
            prim = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0, False])[1] and INTENTOS_BR[k][0] == 1)
            print("="*46)
            print("          CALIFICACIÓN FORMATIVA")
            print("="*46)
            print(f"  Preguntas respondidas:       {resp} de {total}")
            print(f"  Acertadas al primer intento: {prim}")
            print_es(f"  Puntaje ponderado:           {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*46)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*46)
            if resp < total: print(f"  ⚠ Faltan {total-resp} preguntas: cuentan como 0.")
            if nota >= 3.5: print("  ✅ ¡Buen trabajo! Dominas los conceptos.")
            elif nota >= 3.0: print("  🟡 Vas bien, repasa lo que fallaste.")
            else: print("  🔴 Repasa la teoría y vuelve a los puntos.")
            print("\n  Crédito por pregunta: 100 % al primer intento,")
            print("  50 % al segundo, 25 % al tercero. Predice y")
            print("  razona ANTES de verificar: adivinar cuesta nota.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>📊 Tu calificación del taller</h3>"
        "<i>La nota se calcula sobre TODAS las preguntas del taller y premia acertar al primer intento.</i>"),
        boton, salida]))


def esquema():
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(9,4),gridspec_kw={'width_ratios':[1,1.2]})
    # IZQUIERDA: átomos vibrando (frío vs caliente)
    for ax,titulo,amp,col in [(ax1,'',0,'')]:
        pass
    # frío: átomos quietos
    for x in [1,2,3]:
        for y in [1,2,3]:
            ax1.scatter([x],[y],s=120,color='#2980b9',alpha=0.7,zorder=3)
    ax1.annotate('',xy=(3.5,2),xytext=(1,2),arrowprops=dict(arrowstyle='-|>',color='#27ae60',lw=2))
    ax1.text(2,0.3,'FRÍO: átomos quietos\nelectrones pasan fácil (R baja)',ha='center',fontsize=8,color='#2980b9')
    ax1.set_xlim(0,4.5); ax1.set_ylim(0,4); ax1.axis('off'); ax1.set_title('Frío',fontsize=10)
    # gráfico R vs T
    T=np.linspace(0,100,100); R=1+0.004*T
    ax2.plot(T,R,color='#c0392b',lw=2.5)
    ax2.fill_between(T,0,R,color='#c0392b',alpha=0.1)
    ax2.set_xlabel('temperatura (°C)'); ax2.set_ylabel('resistencia R')
    ax2.set_title('R crece con T (átomos vibran más)',fontsize=10)
    ax2.set_ylim(0.9,1.5); ax2.grid(alpha=0.2,linestyle='--')
    ax2.annotate('más calor →\nmás vibración →\nmás choques',xy=(80,1.32),xytext=(30,1.35),fontsize=8,color='#2c3e50',
                 arrowprops=dict(arrowstyle='->',color='#7f8c8d'))
    plt.suptitle('La resistencia aumenta con la temperatura',fontsize=11,fontweight='bold',color='#2c3e50')
    plt.tight_layout(); plt.show()
esquema()

def temperatura(R0=100.0, alpha=0.00390, dT=50.0):
    R=R0*(1+alpha*dT)
    dTs=np.linspace(0,150,100); Rs=R0*(1+alpha*dTs)
    plt.figure(figsize=(9,4.2)); plt.plot(dTs,Rs,lw=2,color='orangered')
    plt.scatter([dT],[R],color='blue',s=60,zorder=5,label=f'ΔT={dT:.0f}°C → R={_3c(R)} Ω')
    plt.xlabel('aumento de temperatura ΔT (°C)'); plt.ylabel('resistencia R (Ω)')
    plt.title('R(T)=R₀(1+αΔT)  ·  base de los sensores RTD'); plt.legend(); plt.grid(alpha=.3)
    plt.tight_layout(); plt.show()


def solucion():
    import re as _re
    def _pc(*a,**k):
        def _f(t):
            if not isinstance(t,str): return t
            _sup='⁰¹²³⁴⁵⁶⁷⁸⁹'
            def _sci(m):
                mant=m.group(1); exp=int(m.group(2))
                e=('⁻' if exp<0 else '')+''.join(_sup[int(d)] for d in str(abs(exp)))
                return mant+'×10'+e
            t=_re.sub(r'(\d+(?:\.\d+)?)[eE]([+-]?\d+)', _sci, t)
            _u=r'(?:s|km/s|km|m/s²|m/s|m|N·m|N|J|W|A|V|T|Ω|C|F|Hz|rad/s²|rad/s|rad|kg·m²/s|kg·m²|kg|Pa|kPa|µF|nC|µC|pF|mWb|Wb|µT|mT|kW|MW|mA|kΩ|MΩ|rpm|°)'
            def _c3(m):
                num=m.group(2); n=int(num)
                if n==0: return m.group(0)
                if len(num)==1: nv=num+'.00'
                elif len(num)==2: nv=num+'.0'
                else: nv=num
                return m.group(1)+nv+m.group(3)
            t=_re.sub(r'(?<![\d.,])([\s=(×])(\d{1,3})(\s'+_u+r'\b)', _c3, t)
            t=_re.sub(r'(\d)\.(\d)', r'\1,\2', t)
            return t
        print(*[_f(x) for x in a], **k)

    R0=100.0; alpha=0.00390; dT=50.0; R=R0*(1+alpha*dT)
    _pc(f"R(T) = R₀(1+αΔT) = 100·(1 + 0,00390·50,0)")
    _pc(f"     = 100·(1 + 0,195) = 100·1,195 = {R:.3g} Ω")
    _pc("Este efecto es la base del termómetro de resistencia (Pt100): a más temperatura, más R.")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 5 ---")
quiz_numerico("Una resistencia de 100 Ω (con coeficiente térmico 0,00390 /°C) se calienta 50,0 °C. Encuentre su nueva resistencia.",120,0.03,"Ω","R=R₀(1+αΔT)≈120 Ω.", id_preg="Corrie_p5_12", peso=3.0)
quiz_opcion_multiple("En un sensor de temperatura tipo RTD (Pt100), al aumentar la temperatura, ¿qué ocurre con su resistencia?",
    ['Disminuye (α < 0 en metales)', 'Aumenta (R = R₀(1 + αΔT))', 'No cambia (R es fija)', 'Se anula (superconduce)'],1,
    "Para metales α>0: más temperatura ⟹ más resistencia. Así mide el sensor.", id_preg="Corrie_p5_13", peso=3.0)

quiz_opcion_multiple("DEFIENDE TU CONCLUSIÓN: un estudiante afirma que un sensor Pt100 mide temperatura porque 'el calor genera un voltaje' en el metal, como un termopar. ¿Correcto o solo convincente?",
    ['Solo convincente: el Pt100 es resistivo, R(T) = R₀(1 + αΔT); no genera voltaje',
     'Correcto: todo sensor de temperatura convierte el calor directamente en voltaje',
     'Correcto: el platino genera un voltaje al calentarse, y por eso se usa también como material en los termopares',
     'Solo convincente: el Pt100 funciona porque su resistencia disminuye al calentarse'],
    0,
    'El Pt100 no genera nada: cambia su resistencia, que AUMENTA con la temperatura (α > 0 en metales). El voltaje por diferencia de temperatura es el efecto Seebeck, propio del termopar.',
    id_preg='Corr_p5_concep_5', peso=2.0)

def _interact_coma(fn, _crudos=(), **controles):
    # Como interact, con lectura de sliders en coma decimal. La salida NO se captura con «with salida:»:
    # se calculan el texto y las figuras y se escriben con append_stdout/append_display_data
    # (evita bloqueos de Voilà en Binder y funciona también en JupyterLite).
    import ipywidgets as _w, io as _io, contextlib as _cl
    import matplotlib.pyplot as _plt
    def _fmt(v, paso, crudo=False):
        # Lectura con coma decimal y 3 cifras significativas (×10ⁿ fuera de 0,001 ≤ |v| < 1000).
        # IntSlider: entero exacto. «crudo»: decimales del paso (slider que representa una medición sin redondear).
        import math
        if isinstance(v, int) and not isinstance(v, bool): return str(v)
        if crudo:
            s = repr(float(paso)); dec = 0 if 'e' in s else len(s.split('.')[1].rstrip('0'))
            return f"{v:.{min(dec, 4)}f}".replace('.', ',').replace('-', '−')
        v = float(v)
        if v == 0 or not math.isfinite(v): return '0' if v == 0 else str(v)
        v = float(f"{v:.2e}")
        if abs(v) >= 1000 or abs(v) < 0.001:
            m, e = f"{v:.2e}".split('e')
            return m.replace('.', ',').replace('-', '−') + '×10' + str(int(e)).translate(str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹'))
        t = f"{v:#.3g}"
        return (t[:-1] if t.endswith('.') else t).replace('.', ',').replace('-', '−')
    filas = []
    for nombre, c in controles.items():
        if isinstance(c, (_w.FloatSlider, _w.IntSlider)):
            c.readout = False
            lab = _w.Label(layout=_w.Layout(min_width='90px'))
            def _act(*_, c=c, lab=lab, cr=(nombre in _crudos)): lab.value = _fmt(c.value, c.step, cr)
            c.observe(_act, 'value'); _act()
            filas.append(_w.HBox([c, lab]))
        else:
            filas.append(c)
    salida = _w.Output()
    def _ejecutar(*_):
        partes, buf = [], _io.StringIO()
        def _vaciar_texto():
            t = buf.getvalue()
            if t: partes.append(('t', t)); buf.seek(0); buf.truncate(0)
        def _capturar(*a, **k):
            _vaciar_texto()
            for num in _plt.get_fignums():
                b = _io.BytesIO(); _plt.figure(num).savefig(b, format='png', bbox_inches='tight')
                partes.append(('i', b.getvalue()))
            _plt.close('all')
        _show = _plt.show; _plt.show = _capturar
        try:
            with _cl.redirect_stdout(buf):
                try: fn(**{k: c.value for k, c in controles.items()})
                except Exception as e: print('⚠ No se pudo calcular con estos valores:', e)
            _capturar()
        finally:
            _plt.show = _show
        _vaciar_texto()
        # Reemplazo en UNA sola asignación: la salida nunca queda vacía (sin saltos de la página)
        import base64 as _b64
        nuevas = []
        for tipo, val in partes:
            if tipo == 't': nuevas.append({'output_type': 'stream', 'name': 'stdout', 'text': val})
            else: nuevas.append({'output_type': 'display_data', 'metadata': {},
                                 'data': {'image/png': _b64.b64encode(val).decode('ascii'), 'text/plain': '<Figure>'}})
        salida.outputs = tuple(nuevas)
    for c in controles.values():
        c.observe(_ejecutar, 'value')
    _ejecutar()
    display(_w.VBox(filas + [salida]))

def _explora_sin_cifras(fn):
    # Opción 3: exploración cualitativa; las cifras de este punto solo están en «Mostrar solución».
    import functools, io, contextlib, re as _re
    _pat = _re.compile(r'(?<![\w.,/])[-−]?\d+(?:[.,]\d+)?(?:\s?×\s?10[⁻⁰¹²³⁴⁵⁶⁷⁸⁹]+|[eE][+-]?\d+)?(?![\wπ·(]|[.,]\d)')
    def _m(t): return _pat.sub('…', t)
    def _ocultar():
        for num in plt.get_fignums():
            fig = plt.figure(num)
            textos = list(fig.texts) + ([fig._suptitle] if getattr(fig, '_suptitle', None) else [])
            for ax in fig.axes:
                textos += [ax.title, ax._left_title, ax._right_title] + list(ax.texts)
                if ax.get_legend(): textos += list(ax.get_legend().get_texts())
            for t in textos: t.set_text(_m(t.get_text()))
    @functools.wraps(fn)
    def envuelta(*a, **k):
        _show = plt.show
        def _show2(*sa, **sk):
            _ocultar(); _show(*sa, **sk)
        buf = io.StringIO(); plt.show = _show2
        try:
            with contextlib.redirect_stdout(buf): fn(*a, **k)
            _ocultar()
        finally:
            plt.show = _show
        for linea in buf.getvalue().splitlines():
            print(linea if ('⚠' in linea or '⛔' in linea) else _m(linea))
        print("(Exploración cualitativa: observa tendencias. Las cifras de este punto están en «Mostrar solución».)")
    return envuelta

print("\n--- 🔬 Explora después de responder ---")
print("Los controles NO arrancan en los datos del enunciado. Primero resuelve y responde;\nluego explora tendencias, casos límite y comprueba tu razonamiento.")
_interact_coma(_explora_sin_cifras(temperatura),
    R0=FloatSlider(value=140.0,min=50,max=200,step=10,description='R₀ (Ω)'),
    alpha=FloatSlider(value=0.0050,min=0.001,max=0.006,step=0.0005,description='α (/°C)',readout_format='.4f'),
    dT=FloatSlider(value=90.0,min=0,max=150,step=10,description='ΔT (°C)'));

---
## ✅ Fin del taller de Corriente y Resistencia (Física BR)
Cada punto nació de un problema real: conductores, protecciones, líneas de transmisión, costo
energético industrial y sensores de temperatura. Experimenta con los sliders y **predice** antes
de pulsar "Mostrar solución".
*Física interactiva para ingeniería — 2026-2.*

In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

try:
    print_es
except NameError:
    def print_es(*args, **kwargs):
        import re as _re
        print(*[_re.sub(r"(\d)\.(\d)", r"\1,\2", a) if isinstance(a, str) else a for a in args], **kwargs)
# Signo menos tipográfico (−) en todo texto impreso: «-5» → «−5» (no toca guiones entre palabras, rangos ni 1e-3)
import builtins as _bi_br
if not getattr(_bi_br.print, '_menos_br', False):
    def _print_br(*a, _orig=_bi_br.print, **k):
        import re as _re
        _orig(*[_re.sub(r'(?<![\w.,\-])-(?=\d)', '−', x) if isinstance(x, str) else x for x in a], **k)
    _print_br._menos_br = True
    _bi_br.print = _print_br
# --- Cuestionarios Física BR v3: opciones barajadas, crédito por intento, nota sobre el total ---
import random as _rnd_br
try:
    PESOS_BR
except NameError:
    PESOS_BR = {}      # id_preg -> peso de cada pregunta mostrada
try:
    INTENTOS_BR
except NameError:
    INTENTOS_BR = {}   # id_preg -> [intentos, resuelta]
_CREDITO_BR = {1: 1.0, 2: 0.5, 3: 0.25}   # crédito según el intento en que se acierta
_MAX_INT_BR = 3

def _evaluar_intento(id_preg, ok, peso, explicacion, msg_ok, msg_mal):
    est = INTENTOS_BR.setdefault(id_preg, [0, False])
    if est[1]:
        print("✔ Ya resolviste esta pregunta; tu crédito quedó registrado.")
        if explicacion: print_es("→ " + explicacion)
        return
    if est[0] >= _MAX_INT_BR:
        print("⛔ Agotaste los 3 intentos con crédito: esta pregunta ya no suma.")
        if explicacion: print_es("→ " + explicacion)
        return
    est[0] += 1
    n = est[0]
    if ok:
        est[1] = True
        cred = _CREDITO_BR.get(n, 0.0)
        REGISTRO_BR[id_preg] = (cred, float(peso))
        print_es(msg_ok + f"  (intento {n}: crédito {int(cred*100)} %)")
        if explicacion: print_es("→ " + explicacion)
    else:
        REGISTRO_BR.setdefault(id_preg, (0.0, float(peso)))
        if n < _MAX_INT_BR:
            print(msg_mal + f"  Intento {n} de {_MAX_INT_BR}. Revisa tu razonamiento antes de volver a intentar.")
        else:
            print(msg_mal + " Agotaste los 3 intentos: esta pregunta ya no suma. Estudia la explicación:")
            if explicacion: print_es("→ " + explicacion)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    orden = list(range(len(opciones))); _rnd_br.shuffle(orden)   # orden distinto en cada sesión
    correcta = orden.index(indice_correcto)
    letras = "abcdefgh"
    etiquetas = [f"{letras[i]}) {opciones[j]}" for i, j in enumerate(orden)]
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=etiquetas, value=None, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            if radio.index is None:
                print("⚠ Elige una opción."); return
            _evaluar_intento(id_preg, radio.index == correcta, peso, explicacion,
                             "✅ ¡Correcto!", "❌ Incorrecto.")
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.strip().replace(",", "."))
            except ValueError: print("⚠ Escribe un número (usa coma o punto decimal)."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            _evaluar_intento(id_preg, ok, peso, explicacion,
                             f"✅ ¡Correcto!  ({valor_correcto:.3g} {unidad})", "❌ Fuera de rango.")
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not PESOS_BR:
                print("No se encontraron cuestionarios en este taller."); return
            peso_total = sum(PESOS_BR.values())
            peso_ok = sum(float(c)*p for c, p in REGISTRO_BR.values())
            nota = 5.0*peso_ok/peso_total if peso_total > 0 else 0.0
            total = len(PESOS_BR)
            resp = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0])[0] > 0)
            prim = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0, False])[1] and INTENTOS_BR[k][0] == 1)
            print("="*46)
            print("          CALIFICACIÓN FORMATIVA")
            print("="*46)
            print(f"  Preguntas respondidas:       {resp} de {total}")
            print(f"  Acertadas al primer intento: {prim}")
            print_es(f"  Puntaje ponderado:           {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*46)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*46)
            if resp < total: print(f"  ⚠ Faltan {total-resp} preguntas: cuentan como 0.")
            if nota >= 3.5: print("  ✅ ¡Buen trabajo! Dominas los conceptos.")
            elif nota >= 3.0: print("  🟡 Vas bien, repasa lo que fallaste.")
            else: print("  🔴 Repasa la teoría y vuelve a los puntos.")
            print("\n  Crédito por pregunta: 100 % al primer intento,")
            print("  50 % al segundo, 25 % al tercero. Predice y")
            print("  razona ANTES de verificar: adivinar cuesta nota.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>📊 Tu calificación del taller</h3>"
        "<i>La nota se calcula sobre TODAS las preguntas del taller y premia acertar al primer intento.</i>"),
        boton, salida]))


# Calificación formativa del taller
calificacion_final(total_preguntas=18)
